In [19]:
#!/usr/bin/env python3
"""
Protein Tunnel Detection using a Weighted Voronoi Diagram via Lifting
with a refined boundary.

This module computes an additively weighted (power) Voronoi diagram for
protein atom positions (with weights, e.g. linear van der Waals radii)
using the lifting method with SciPy’s ConvexHull. It then constructs a graph
from the weighted Voronoi vertices, searches for tunnel‐like paths from a
starting interior point to the solvent region as defined by a refined boundary.
The boundary is obtained by computing the convex hull of the protein and then,
for each hull vertex, inscribing the maximum sphere that does not intersect
the protein’s VDW surfaces. A candidate Voronoi vertex is deemed solvent
accessible if it lies within one of these spheres.
The pipeline computes tunnel metrics, clusters the tunnels, and writes out
the tunnel “center‐lines” as PDB files (with clearance in the B‑factor column).

Requirements:
  • MDAnalysis
  • numpy, pandas
  • scipy
  • networkx
  • scikit‑learn
  • (Optional) matplotlib for plotting

Author: [Your Name]
Date: [Date]
"""

import numpy as np
import pandas as pd
import MDAnalysis as mda
from scipy.spatial import ConvexHull, cKDTree
import networkx as nx
from sklearn.cluster import AgglomerativeClustering
import sys
import os

# ---------------- Weighted Voronoi via Lifting ----------------

def weighted_voronoi_3d(points, weights):
    """
    Compute the additively weighted (power) Voronoi vertices for 3D points
    using the lifting method.
    
    Each point p in R^3 with weight w is lifted to 
      (p, ||p||^2 - w) in R^4.
    
    The convex hull of these lifted points is computed. Facets on the
    lower envelope (i.e. those with a positive coefficient for the lifted
    coordinate) yield, via duality, vertices in R^3. Specifically, for a 
    facet with hyperplane equation:
    
        a0*x + a1*y + a2*z + a3*z4 + d = 0,
    
    its dual vertex is given by:
    
        v = - (a0, a1, a2) / (2*a3).
    
    Parameters:
        points: (N,3) array of 3D coordinates.
        weights: (N,) array of weights (e.g. van der Waals radii).
        
    Returns:
        dual_points: (M,3) array of weighted Voronoi (power) vertices.
    """
    N, d = points.shape
    if d != 3:
        raise ValueError("This function only works for 3D points.")
    
    norms_sq = np.sum(points**2, axis=1, keepdims=True)
    lifted = np.hstack([points, norms_sq - weights.reshape(-1, 1)])
    hull = ConvexHull(lifted)
    
    dual_points = []
    tol = 1e-12
    for eq in hull.equations:
        # eq is [a0, a1, a2, a3, d]
        if eq[3] > tol:
            dual = -eq[:3] / (2 * eq[3])
            dual_points.append(dual)
    return np.array(dual_points)

# ---------------- Boundary Refinement via Inscribed Spheres ----------------

def add_bulk_points(prot_coords, margin=5.0, n_points=100):
    """
    Add bulk-solvent points distributed uniformly on a sphere that encloses the protein.
    
    Parameters:
        prot_coords: (N,3) array of protein atom coordinates.
        margin: additional distance (in Å) to add to the protein's maximal extent.
        n_points: number of points to generate on the sphere.
        
    Returns:
        bulk_points: (n_points,3) array of bulk points.
    """
    center = np.mean(prot_coords, axis=0)
    # Compute the maximum distance from the center
    dists = np.linalg.norm(prot_coords - center, axis=1)
    R = np.max(dists) + margin
    # Generate points uniformly on a sphere (using spherical coordinates)
    phi = np.arccos(1 - 2 * np.random.rand(n_points))
    theta = 2 * np.pi * np.random.rand(n_points)
    x = R * np.sin(phi) * np.cos(theta) + center[0]
    y = R * np.sin(phi) * np.sin(theta) + center[1]
    z = R * np.cos(phi) + center[2]
    bulk_points = np.vstack((x, y, z)).T
    return bulk_points

def refine_boundary(hull_points, prot_coords, prot_vdw, use_uniform=True):
    """
    Given a set of convex hull vertices (from an augmented set of points)
    and the protein coordinates with their van der Waals radii, compute for each hull
    vertex the maximal inscribed sphere (the largest sphere that can be placed
    at that vertex without intersecting the protein's VDW surfaces).
    
    If use_uniform is True, then all protein atoms are assumed to have the same VDW
    radius equal to the minimum of prot_vdw.
    
    Returns a list of tuples (center, clearance).
    """
    if use_uniform:
        effective_r = np.min(prot_vdw)
    tree = cKDTree(prot_coords)
    spheres = []
    for v in hull_points:
        if use_uniform:
            dists = np.linalg.norm(prot_coords - v, axis=1) - effective_r
        else:
            dists = np.linalg.norm(prot_coords - v, axis=1) - prot_vdw
        clearance = np.min(dists)
        clearance = max(clearance, 0.0)
        spheres.append((v, clearance))
    return spheres

def identify_boundary_nodes_with_spheres(valid_points, spheres):
    """
    Mark a candidate Voronoi vertex as solvent accessible if it lies
    inside any of the inscribed spheres.
    """
    indices = []
    for i, pt in enumerate(valid_points):
        for center, radius in spheres:
            if np.linalg.norm(pt - center) <= radius:
                indices.append(i)
                break
    return indices

# ---------------- Graph Construction from Dual Points ----------------

def build_nn_graph(points, k=5):
    """
    Build an undirected graph by connecting each point to its k nearest neighbors.
    
    Parameters:
       points: (M,3) array of 3D coordinates (weighted Voronoi vertices).
       k: number of nearest neighbors (default=5).
       
    Returns:
       G: networkx.Graph with node attribute 'pos' storing the coordinate.
    """
    G = nx.Graph()
    tree = cKDTree(points)
    for i, pt in enumerate(points):
        G.add_node(i, pos=pt)
        dists, indices = tree.query(pt, k=k+1)
        for j in indices[1:]:
            if not G.has_edge(i, j):
                dist = np.linalg.norm(pt - points[j])
                G.add_edge(i, j, weight=dist)
    return G

# ---------------- Tunnel Detection Pipeline ----------------

def compute_convex_hull(points):
    """Compute the convex hull of points and return hull and hull vertices."""
    hull = ConvexHull(points)
    hull_points = points[hull.vertices]
    return hull, hull_points

def find_shortest_paths(G, valid_points, start_pt, boundary_indices):
    """
    Find candidate tunnel paths from the node closest to start_pt to each node
    in boundary_indices.
    """
    node_list = list(G.nodes())
    node_positions = np.array([G.nodes[n]['pos'] for n in node_list])
    tree = cKDTree(node_positions)
    dist, pos_idx = tree.query(start_pt)
    start_node = node_list[pos_idx]
    paths = []
    for b in boundary_indices:
        try:
            path = nx.shortest_path(G, source=start_node, target=b, weight='weight')
            paths.append(path)
        except nx.NetworkXNoPath:
            continue
    return start_node, paths

def compute_path_metrics(path, node_positions, protein_coords, protein_vdw):
    """
    Compute tunnel metrics along a given path: total length, minimum clearance (bottleneck),
    and average clearance squared (throughput).
    Clearance is computed as distance from the node to the nearest protein atom minus that atom's VDW radius.
    """
    positions = np.array([node_positions[n] for n in path])
    length = np.sum(np.linalg.norm(np.diff(positions, axis=0), axis=1))
    protein_tree = cKDTree(protein_coords)
    clearances = []
    for pos in positions:
        d, idx = protein_tree.query(pos)
        clear = d - protein_vdw[idx]
        clearances.append(clear)
    clearances = np.array(clearances)
    bottleneck = np.min(clearances)
    throughput = np.mean(clearances**2)
    return length, bottleneck, throughput

def rank_tunnels(paths, G, protein_coords, protein_vdw):
    """
    For each candidate tunnel (path), compute metrics and return a sorted list by throughput.
    """
    node_positions = np.array([G.nodes[n]['pos'] for n in G.nodes()])
    tunnel_data = []
    for path in paths:
        length, bottleneck, throughput = compute_path_metrics(path, node_positions, protein_coords, protein_vdw)
        tunnel_data.append({
            'path': path,
            'length': length,
            'bottleneck': bottleneck,
            'throughput': throughput
        })
    tunnel_data = sorted(tunnel_data, key=lambda t: t['throughput'], reverse=True)
    return tunnel_data

def cluster_tunnels(tunnel_data, G, n_clusters=2):
    """
    Cluster tunnels using features derived from their centroid and metrics.
    If only one tunnel is found, assign it to cluster 0.
    """
    if len(tunnel_data) == 1:
        tunnel_data[0]['cluster'] = 0
        return tunnel_data, None, np.array([0])
    features = []
    for t in tunnel_data:
        path = t['path']
        pos = np.array([G.nodes[n]['pos'] for n in path])
        centroid = pos.mean(axis=0)
        feat = np.concatenate([centroid, [t['length'], t['bottleneck'], t['throughput']]])
        features.append(feat)
    features = np.array(features)
    clustering = AgglomerativeClustering(n_clusters=n_clusters)
    labels = clustering.fit_predict(features)
    for i, t in enumerate(tunnel_data):
        t['cluster'] = labels[i]
    return tunnel_data, features, labels

def write_tunnel_pdb(tunnel, G, protein_universe, filename):
    """
    Write a PDB file representing the tunnel path.
    Each node is written as an ATOM record with its clearance (from protein) in the B-factor.
    """
    protein_coords = protein_universe.select_atoms('all').positions
    protein_tree = cKDTree(protein_coords)
    lines = []
    path = tunnel['path']
    for i, n in enumerate(path):
        pos = G.nodes[n]['pos']
        d, idx = protein_tree.query(pos)
        clearance = d  # optionally subtract protein_vdw if desired
        line = "ATOM  {:5d}  CA  TUN A{:4d}    {:8.3f}{:8.3f}{:8.3f}{:6.2f}{:6.2f}".format(
            i+1, i+1, pos[0], pos[1], pos[2], 1.00, clearance)
        lines.append(line)
    lines.append("END")
    with open(filename, 'w') as f:
        f.write("\n".join(lines))
    print(f"Wrote tunnel PDB: {filename}")

# ---------------- Main Pipeline Function ----------------

def find_tunnels(universe, start_point, protein_sel='protein',
                 vdw_attr='vdw_radius', default_vdw=1.5, cluster_n=2,
                 boundary_tol=1.0, verbose=True):
    """
    Find tunnels in a protein using a weighted Voronoi (power) diagram with
    a refined boundary. The refinement is performed by computing the convex
    hull of the protein and then, for each hull vertex, inscribing the largest
    sphere that does not intersect the protein's van der Waals surfaces.
    A candidate weighted Voronoi vertex is considered solvent accessible if
    it lies within any of these spheres.
    
    Parameters:
      universe: MDAnalysis Universe with the protein.
      start_point: 3D coordinate (numpy array) inside the protein.
      protein_sel: Selection string for protein atoms.
      vdw_attr: Attribute name for van der Waals radius (default 'vdw_radius').
      default_vdw: Default VDW radius if not provided.
      cluster_n: Number of clusters for grouping tunnels.
      boundary_tol: Tolerance (Å) to consider a candidate point as accessible.
      verbose: If True, print progress.
      
    Returns:
      tunnel_df: pandas DataFrame summarizing tunnel metrics.
      tunnel_data: List of tunnel dictionaries.
      G: Graph of weighted Voronoi dual vertices.
      dual_vertices: Array of weighted Voronoi vertices.
    """
    if verbose:
        print("Selecting protein atoms...")
    prot_atoms = universe.select_atoms(protein_sel)
    prot_coords = prot_atoms.positions
    
    # Retrieve VDW radii; if not available, use default.
    try:
        vdw_radii = prot_atoms.__dict__.get(vdw_attr, None)
        if vdw_radii is None:
            vdw_radii = np.array([getattr(a, vdw_attr, default_vdw) for a in prot_atoms])
        else:
            vdw_radii = np.array(vdw_radii)
    except Exception:
        vdw_radii = np.full(len(prot_coords), default_vdw)
    
    weights = vdw_radii  # linear weight
    
    if verbose:
        print("Computing weighted Voronoi diagram via lifting...")
    dual_vertices = weighted_voronoi_3d(prot_coords, weights)
    if verbose:
        print(f"Computed {len(dual_vertices)} weighted Voronoi vertices.")
    
    if verbose:
        print("Building nearest-neighbor graph on weighted Voronoi vertices...")
    G = build_nn_graph(dual_vertices, k=5)
    
        # Compute convex hull of protein atoms.
    if verbose:
        print("Computing convex hull of protein atoms...")
    # Compute convex hull of protein atoms combined with bulk solvent points.
    bulk_pts = add_bulk_points(prot_coords, margin=5.0, n_points=100)
    augmented_coords = np.vstack((prot_coords, bulk_pts))
    hull, hull_points = compute_convex_hull(augmented_coords)

    # Now refine the boundary by computing the maximal inscribed spheres at each hull vertex.
    spheres = refine_boundary(hull_points, prot_coords, vdw_radii, use_uniform=True)

    # Identify which weighted Voronoi (dual) vertices lie within at least one of these spheres.
    boundary_indices = identify_boundary_nodes_with_spheres(dual_vertices, spheres)

    if len(boundary_indices) == 0:
        raise ValueError("No boundary nodes found; consider adjusting boundary_tol.")
    
    if verbose:
        print("Finding candidate tunnel paths via shortest path search...")
    start_node, paths = find_shortest_paths(G, dual_vertices, start_point, boundary_indices)
    if verbose:
        print(f"Found {len(paths)} candidate tunnel paths.")
    if not paths:
        raise ValueError("No candidate tunnels found from the given start point.")
    
    tunnel_data = rank_tunnels(paths, G, prot_coords, vdw_radii)
    
    # If only one tunnel found, assign it to cluster 0.
    if len(tunnel_data) == 1:
        tunnel_data[0]['cluster'] = 0
    else:
        tunnel_data, features, labels = cluster_tunnels(tunnel_data, G, n_clusters=cluster_n)
    
    # Create pandas DataFrame summary.
    rows = []
    for i, t in enumerate(tunnel_data):
        rows.append({
            'tunnel_id': i,
            'length': t['length'],
            'bottleneck': t['bottleneck'],
            'throughput': t['throughput'],
            'cluster': t.get('cluster', 0),
            'path': t['path']
        })
    tunnel_df = pd.DataFrame(rows).sort_values(by='throughput', ascending=False).reset_index(drop=True)
    return tunnel_df, tunnel_data, G, dual_vertices

# ---------------- Main Script ----------------

if __name__ == '__main__':
    pdb_file = "1be0.pdb"
    try:
        u = mda.Universe(pdb_file)
    except Exception as e:
        print("Error loading PDB file:", e)
        sys.exit(1)
    
    start_pt = np.array([28.627,  32.479,  30.190])
    print("Using start point:", start_pt)
    
    tunnel_df, tunnel_data, G, dual_vertices = find_tunnels(u, start_pt, verbose=True)
    
    print("Tunnel Summary:")
    print(tunnel_df[['tunnel_id', 'length', 'bottleneck', 'throughput', 'cluster']])
    
    best_cluster = tunnel_df['cluster'].mode()[0]
    best_tunnels = tunnel_df[tunnel_df['cluster'] == best_cluster]
    for i, row in best_tunnels.iterrows():
        tid = row['tunnel_id']
        out_filename = f"weighted_tunnel_{tid}.pdb"
        write_tunnel_pdb(tunnel_data[tid], G, u, out_filename)


Using start point: [28.627 32.479 30.19 ]
Selecting protein atoms...
Computing weighted Voronoi diagram via lifting...
Computed 166 weighted Voronoi vertices.
Building nearest-neighbor graph on weighted Voronoi vertices...
Computing convex hull of protein atoms...
Finding candidate tunnel paths via shortest path search...
Found 21 candidate tunnel paths.
Tunnel Summary:
    tunnel_id      length  bottleneck    throughput  cluster
0           0  863.014262   -0.271595  57122.212096        1
1           1  599.419446    1.004422  17792.608456        0
2           2  773.799667    1.004422  12954.018695        0
3           3  781.488580    1.004422  12928.356458        0
4           4  525.128250    1.004422   3754.901195        0
5           5  348.064597    0.543493   3584.319189        0
6           6  454.200353    0.543493   3144.980732        0
7           7  249.466887    1.004422   2868.325998        0
8           8  308.285050    1.004422   2532.255342        0
9           9  28